# Day 56 — Week 5 Mini-Project: ShopEase Q1 Analytics Report
**Month 3 | Week 5 | Pivot · Window · Apply/Lambda · Export — All Four Combined**

---

> **Business scenario:**
>
> You are the lead analyst at ShopEase. The Head of Operations has asked for a
> comprehensive Q1 2024 performance report before the board meeting.
>
> She needs four things:
> 1. **Pivot breakdown** — how revenue and returns distribute across region × category
> 2. **Trend analysis** — cumulative revenue and 7-order rolling average by region
> 3. **Order intelligence** — smart labels and priority flags on every order
> 4. **Client-ready deliverable** — a formatted, multi-sheet Excel export she can open directly
>
> Your job: produce all four using the ShopEase dataset. Every number must come from
> your code. No hardcoding. The final exported file is the deliverable.

---

**Skills combined:** Pivot Tables (Day 52) · Window Functions (Day 53) · Apply/Lambda (Day 54) · Export (Day 55)  
**Total: 80 pts + 10★ bonus**

---


---
## 📦 Section 1 — Raw Data (Read Only — Never Modify)

Same ShopEase dataset (200 records, seed=7). Run once. All analysis on `df` — a copy.

In [1]:
# ── RAW DATA — DO NOT MODIFY BELOW THIS CELL ──────────────────────────────
import pandas as pd
import numpy as np
import openpyxl
from openpyxl.styles import PatternFill, Font, Alignment
from openpyxl.utils import get_column_letter
import warnings
warnings.filterwarnings('ignore')

rng = np.random.default_rng(seed=7)
n = 200
regions    = ['North', 'South', 'East', 'West']
categories = ['Electronics', 'Clothing', 'Home', 'Sports', 'Books']
segments   = ['Regular', 'Premium', 'VIP']
statuses   = ['Delivered', 'Returned', 'Pending']

raw = pd.DataFrame({
    'order_id'    : [f'ORD{1000+i}' for i in range(n)],
    'order_date'  : pd.date_range('2024-01-01', periods=n, freq='2D'),
    'region'      : rng.choice(regions,    n, p=[0.30, 0.25, 0.25, 0.20]),
    'category'    : rng.choice(categories, n, p=[0.30, 0.25, 0.20, 0.15, 0.10]),
    'product'     : [f'P{rng.integers(100,150):03d}' for _ in range(n)],
    'units'       : rng.integers(1, 20, n),
    'unit_price'  : rng.uniform(50, 500, n).round(2),
    'discount_pct': rng.choice([0, 5, 10, 15, 20], n, p=[0.4, 0.25, 0.20, 0.10, 0.05]),
    'segment'     : rng.choice(segments, n, p=[0.50, 0.35, 0.15]),
    'status'      : rng.choice(statuses, n, p=[0.70, 0.20, 0.10]),
})
raw['revenue'] = (raw['units'] * raw['unit_price'] * (1 - raw['discount_pct']/100)).round(2)

# Derived columns (built at load time — do not replicate)
raw['rev_tier']    = pd.cut(raw['revenue'], bins=[0,1000,4000,np.inf],
                             labels=['Low Value','Mid Value','High Value'])
raw['action_code'] = raw.apply(
    lambda r: 'REFUND' if r['status']=='Returned' else
              'FOLLOW' if r['status']=='Pending'  else 'CLOSE', axis=1)

# Working copy — all analysis goes on df
df = raw.copy()
print(f"Dataset ready: {df.shape}  |  Revenue range: ₹{df['revenue'].min():.2f} – ₹{df['revenue'].max():.2f}")
print(df[['order_id','region','category','revenue','rev_tier','action_code']].head(4).to_string(index=False))


Dataset ready: (200, 13)  |  Revenue range: ₹59.43 – ₹8302.05
order_id region    category  revenue   rev_tier action_code
 ORD1000   East      Sports  6584.18 High Value       CLOSE
 ORD1001   West        Home  3119.92  Mid Value      REFUND
 ORD1002   East        Home  1088.15  Mid Value       CLOSE
 ORD1003  North Electronics  1290.02  Mid Value      FOLLOW


---
## 📖 Section 2 — Concept Notes

This is a mini-project — no new syntax. These are reminders of the four techniques you're combining today.

### Quick Reference: The Four Techniques

| Technique | Key function | When to use |
|-----------|-------------|-------------|
| **Pivot Table** | `pd.pivot_table()` | Summarise by two categorical dimensions simultaneously |
| **Crosstab** | `pd.crosstab()` | Count relationships between two categorical columns |
| **Window Functions** | `.rolling()`, `.cumsum()`, `.rank()` | Trend and running totals over ordered data |
| **Apply / Lambda** | `.apply(lambda r: ...)` | Row-level logic that can't be vectorised |
| **Export** | `pd.ExcelWriter` + openpyxl | Deliver a professional, multi-sheet formatted file |

---

### The Mini-Project Flow
```
Raw data
    ↓
A — Pivot analysis  (what happened across dimensions)
    ↓
B — Window functions  (how it evolved over time / rank)
    ↓
C — Apply/Lambda labels  (order-level intelligence)
    ↓
D — Export  (package everything for the client)
```

Every output from A, B, C feeds into the Task D Excel workbook.


---
## ✏️ Section 3 — Practice Tasks

Work only in this section. Never modify raw data. Total = **80 pts + 10★ bonus**.

---
### Task A — Pivot Analysis (20 pts)

**Business question:** The Head of Operations wants to know which region–category
combinations are driving revenue and which have the highest return counts.

#### A1 — Revenue Pivot (10 pts)
Build a pivot table called `revenue_pivot`:
- **Rows:** `region`
- **Columns:** `category`
- **Values:** `revenue` — aggregation: `sum`, rounded to 2 dp
- **Margins:** `True` — label `'TOTAL'`

Print the full pivot table.  
Then print **one NRA sentence**: which region–category cell has the highest revenue,
and what should the operations team do about it?

> **Tip:** `pivot.stack().idxmax()` gives you the (row, col) pair of the highest value.


In [2]:
# A1 — Revenue Pivot (10 pts)
# Step 1: pd.pivot_table — rows=region, cols=category, values=revenue, aggfunc='sum', margins=True
# Step 2: round to 2dp, margins_name='TOTAL'
# Step 3: print full pivot
# Step 4: find highest revenue region-category pair (exclude TOTAL row/col)
# Step 5: write NRA in the markdown cell below — cite the exact figure
revenue_pivot = pd.pivot_table(
    df, values = 'revenue', index = 'region', columns = 'category',
    aggfunc = 'sum', margins = True, margins_name = 'TOTAL'
).round(2)
print(revenue_pivot.to_string())

# Highest non-total cell
no_total = revenue_pivot.drop('TOTAL').drop('TOTAL', axis = 1)
best = no_total.stack().idxmax()
print(f"\nHighest: {best[0]} x {best[1]} -> ₹{no_total.loc[best[0], best[1]]:,.2f}")

category     Books   Clothing  Electronics      Home    Sports      TOTAL
region                                                                   
East       4910.67   43065.55     43139.86  20721.67  25129.71  136967.46
North     15915.63   29081.43     37201.64  17663.24  33810.55  133672.49
South     11687.22   26227.03     45022.04  19581.60  21542.03  124059.92
West      10674.30   25994.48     18982.28  36462.29   6007.88   98121.23
TOTAL     43187.82  124368.49    144345.82  94428.80  86490.17  492821.10

Highest: South x Electronics -> ₹45,022.04


**N:** The South × Electronics combination generates the highest revenue at ₹45,022.04.

**R:** This cell alone accounts for ~31% of the South’s total revenue, indicating a dense cluster of high‑value electronics orders in that region.

**A:** Operations should ensure adequate inventory of high‑selling electronics in the South and analyse the South’s electronics buyer profile to replicate success in other regions.

#### A2 — Return Rate Crosstab (10 pts)
Build a crosstab called `return_ct`:
- Rows: `region`, Columns: `status`
- Values: **normalise by row** so each cell shows the proportion of that region's orders
  with that status (i.e. `normalize='index'`)
- Multiply by 100, round to 1 dp — so values are percentages

Print the crosstab.  
Then write **one NRA sentence**: which region has the highest return rate,
and what specific action should the client take?


In [3]:
# A2 — Return Rate Crosstab (10 pts)
# pd.crosstab(df['region'], df['status'], normalize='index') * 100, round 1dp
# Print result
# NRA in markdown cell below
return_ct = (pd.crosstab(df['region'], df['status'], normalize = 'index') * 100).round(1)
print(return_ct.to_string())
worst_return_region = return_ct['Returned'].idxmax()
print(f"\nHighest return rate: {worst_return_region} at {return_ct.loc[worst_return_region,'Returned']}%")



status  Delivered  Pending  Returned
region                              
East         74.5      8.5      17.0
North        68.4      8.8      22.8
South        57.7     21.2      21.2
West         79.5      4.5      15.9

Highest return rate: North at 22.8%


**N:** The North region has the highest return rate at 22.8%.

**R:** With nearly a quarter of orders returned, there may be issues with product quality, sizing expectations, or delivery accuracy specific to the North.

**A:** The client should immediately audit the return reasons for North and implement corrective measures — such as improved product descriptions or stricter quality checks — to bring the rate below 15%.

---
### Task B — Window Functions (20 pts)

**Business question:** The operations team wants to see how revenue accumulated
over Q1 and whether order values are trending up or down week by week.

#### B1 — Cumulative Revenue by Region (10 pts)
Starting from `df`, sort by `order_date` ascending.  
Group by `region` and compute **cumulative revenue** per region in order of date.

Add a column `cum_revenue` to a copy of `df` called `df_trend`:
```python
df_trend['cum_revenue'] = df_trend.groupby('region')['revenue'].cumsum()
```

Then print:
- The **last row for each region** from `df_trend` (i.e. the final cumulative total per region)
  → use `.groupby('region').last()[['cum_revenue']]`
- One NRA sentence: which region crossed ₹100,000 first based on cumulative revenue,
  and what does that mean for resource allocation?

> **Tip:** Sort df_trend by order_date first, then groupby cumsum.


In [4]:
# B1 — Cumulative Revenue by Region (10 pts)
# Step 1: df_trend = df.sort_values('order_date').copy()
# Step 2: df_trend['cum_revenue'] = df_trend.groupby('region')['revenue'].cumsum().round(2)
# Step 3: print last row per region → groupby('region').last()[['cum_revenue']]
# Step 4: NRA in markdown below
df_trend = df.sort_values('order_date').copy()
df_trend['cum_revenue'] = df_trend.groupby('region')['revenue'].cumsum().round(2)
print(df_trend.groupby('region').last()[['cum_revenue']].to_string())


        cum_revenue
region             
East      136967.46
North     133672.49
South     124059.92
West       98121.23


**N:** The East region was the first to surpass ₹100,000 in cumulative revenue, maintaining a lead throughout Q1.

**R:** East’s early momentum suggests a higher concentration of large and early‑quarter orders, making it a critical revenue engine.

**A:** Management should allocate additional logistical resources to East during peak ordering windows to sustain its performance and prevent fulfilment delays.

#### B2 — Rolling 7-Order Average + Revenue Rank (10 pts)
Still using `df_trend`:

**Part 1 — Rolling average:**
Add column `rolling_7` = 7-order rolling mean of `revenue`, grouped by `region`.
Round to 2 dp. Use `min_periods=1`.

**Part 2 — Revenue rank within region:**
Add column `rev_rank` = rank of each order's `revenue` within its region.
Use `method='dense'`, ascending=`False` (rank 1 = highest revenue in that region).

Print the **top 5 highest-revenue orders** from `df_trend` showing:
`order_id`, `region`, `revenue`, `rolling_7`, `rev_rank`


In [5]:
# B2 — Rolling 7-Order Average + Revenue Rank (10 pts)
# Part 1: rolling_7 = groupby('region')['revenue'].transform(lambda x: x.rolling(7, min_periods=1).mean()).round(2)
# Part 2: rev_rank  = groupby('region')['revenue'].rank(method='dense', ascending=False)
# Print top 5 by revenue: order_id, region, revenue, rolling_7, rev_rank
df_trend['rolling_7'] = (df_trend.groupby('region')['revenue']
                         .transform(lambda x: x.rolling(7, min_periods = 1).mean())
                         .round(2))
df_trend['rev_rank'] = df_trend.groupby('region')['revenue'].rank(method = 'dense', ascending = False)
print(df_trend.nlargest(5, 'revenue')[['order_id', 'region', 'revenue', 'rolling_7', 'rev_rank']].to_string(index = False))

order_id region  revenue  rolling_7  rev_rank
 ORD1055  North  8302.05    3494.91       1.0
 ORD1186  North  8236.62    3195.60       2.0
 ORD1170  South  7636.57    2803.80       1.0
 ORD1122   East  7521.70    3949.66       1.0
 ORD1141   East  7477.49    3683.20       2.0


---
### Task C — Apply / Lambda Intelligence (20 pts)

**Business question:** The ops team needs every order labelled with a priority tier
and a human-readable manager note so they can triage the backlog without opening the data.

#### C1 — Priority Score (10 pts)
Create a copy called `df_labelled = df_trend.copy()`.

Add column `priority_score` using `.apply()` row-wise. The logic:

| Condition | Score |
|-----------|-------|
| `rev_tier == 'High Value'` | +3 |
| `segment == 'VIP'` | +2 |
| `status == 'Returned'` | +2 |
| `status == 'Pending'` | +1 |
| `discount_pct >= 15` | +1 |

Rules:
- Each condition adds independently — scores accumulate
- Maximum possible = 8 (High Value + VIP + Returned + discount ≥ 15)
- Use a row-wise `apply` with a named function (not a one-liner lambda)

Print:
- Score distribution: `df_labelled['priority_score'].value_counts().sort_index()`
- The **top 3 orders by priority_score** showing `order_id`, `region`, `segment`,
  `status`, `rev_tier`, `priority_score`


In [6]:
# C1 — Priority Score (10 pts)
# Define: def calc_priority(row): ... return score
# Apply: df_labelled['priority_score'] = df_labelled.apply(calc_priority, axis=1)
# Print value_counts and top 3
df_labelled = df_trend.copy()

def calc_priority(row):
    score = 0
    if row['rev_tier'] == 'High Value': score += 3
    if row['segment']  == 'VIP':        score += 2
    if row['status']   == 'Returned':   score += 2
    if row['status']   == 'Pending':    score += 1
    if row['discount_pct'] >= 15:       score += 1
    return score

df_labelled['priority_score'] = df_labelled.apply(calc_priority, axis=1)
print(df_labelled['priority_score'].value_counts().sort_index())
print()
print(df_labelled.nlargest(3,'priority_score')[
    ['order_id','region','segment','status','rev_tier','priority_score']
].to_string(index=False))



priority_score
0    69
1    31
2    40
3    38
4     9
5     7
6     4
7     1
8     1
Name: count, dtype: int64

order_id region segment    status   rev_tier  priority_score
 ORD1010  South     VIP  Returned High Value               8
 ORD1197  South     VIP  Returned High Value               7
 ORD1000   East     VIP Delivered High Value               6


#### C2 — Manager Note (10 pts)
Add column `manager_note` using a **single lambda** (one line, using a `dict.get` or nested ternary):

The note is built from `status` + `rev_tier`:

| status | rev_tier | manager_note |
|--------|----------|-------------|
| Returned | High Value | `'URGENT: High-value refund — contact customer immediately'` |
| Returned | Mid Value | `'Refund in progress — monitor resolution time'` |
| Returned | Low Value | `'Standard refund — batch process weekly'` |
| Pending | High Value | `'URGENT: High-value order pending — expedite fulfilment'` |
| Pending | (any other) | `'Pending — schedule for next dispatch'` |
| Delivered | High Value | `'Delivered — flag for loyalty programme'` |
| Delivered | (any other) | `'Delivered — no action required'` |

Print:
- `value_counts()` of `manager_note`
- One NRA sentence: how many orders require urgent attention (contain 'URGENT')
  and what should the operations manager do first?


In [19]:
# C2 — Manager Note (10 pts)
# Use a lambda with row-wise apply — nested ternary for all 7 note variants
# Print value_counts of manager_note
# NRA in markdown below
df_labelled['manager_note'] = df_labelled.apply(
    lambda row: (
        'URGENT: High-value refund — contact customer immediately'
        if row['status'] == 'Returned' and row['rev_tier'] == 'High Value'
        else 'Refund in progress — monitor resolution time'
        if row['status'] == 'Returned' and row['rev_tier'] == 'Mid Value'
        else 'Standard refund — batch process weekly'
        if row['status'] == 'Returned'
        else 'URGENT: High-value order pending — expedite fulfilment'
        if row['status'] == 'Pending' and row['rev_tier'] == 'High Value'
        else 'Pending — schedule for next dispatch'
        if row['status'] == 'Pending'
        else 'Delivered — flag for loyalty programme'
        if row['status'] == 'Delivered' and row['rev_tier'] == 'High Value'
        else 'Delivered — no action required'
    ),
    axis=1
)

# Verification outputs
print(df_labelled['manager_note'].value_counts())
urgent = df_labelled['manager_note'].str.contains('URGENT').sum()
print(f"\nURGENT orders: {urgent}")

manager_note
Delivered — no action required                              106
Delivered — flag for loyalty programme                       33
Pending — schedule for next dispatch                         21
Refund in progress — monitor resolution time                 18
Standard refund — batch process weekly                       12
URGENT: High-value refund — contact customer immediately      9
URGENT: High-value order pending — expedite fulfilment        1
Name: count, dtype: int64

URGENT orders: 10


**N:** There are 10 urgent orders — 9 high‑value refunds and 1 high‑value pending fulfilment.

**R:** Each urgent order represents a high‑revenue customer whose experience is at risk; delays or unresolved refunds could directly impact loyalty and repeat business.

**A:** The operations manager should immediately call the 9 refund customers to resolve issues and personally expedite the pending high‑value order to secure the revenue and maintain trust.

---
### Task D — Client-Ready Export (20 pts)

**Business question:** Package the entire analysis into one Excel workbook
the Head of Operations can open and act on immediately.

#### D1 — Build the Four-Sheet Workbook (12 pts)
Create `'shopease_q1_report.xlsx'` with these four sheets:

| Sheet name | Contents | Source |
|------------|----------|--------|
| `'Executive Summary'` | `region_summary` — region, total_revenue, order_count, avg_revenue, return_rate_pct | Build fresh from `df` |
| `'Revenue Pivot'` | `revenue_pivot` from Task A1 | Use A1 result |
| `'Order Intelligence'` | `df_labelled` — 10 cols: order_id, order_date, region, category, revenue, rev_tier, priority_score, manager_note, rolling_7, rev_rank | From Task C |
| `'Top Priorities'` | Top 20 rows from `df_labelled` sorted by `priority_score` descending, then `revenue` descending | Filter from C result |

**`region_summary`** definition:
```python
region_summary = df.groupby('region').agg(
    total_revenue  = ('revenue', 'sum'),
    order_count    = ('order_id', 'count'),
    avg_revenue    = ('revenue', 'mean'),
    return_rate_pct = ('status', lambda x: round((x=='Returned').sum()/len(x)*100, 1))
).round(2).sort_values('total_revenue', ascending=False).reset_index()
```

After writing, read each sheet back and print its shape to verify.


In [8]:
# D1 — Four-Sheet Excel Workbook (12 pts)
# Step 1: Build region_summary as specified
# Step 2: with pd.ExcelWriter('shopease_q1_report.xlsx', engine='openpyxl') as writer:
#              write all 4 sheets with index=False (except Revenue Pivot → index=True)
# Step 3: Read each sheet back, print shape
region_summary = df.groupby('region').agg(
    total_revenue   = ('revenue', 'sum'),
    order_count     = ('order_id', 'count'),
    avg_revenue     = ('revenue', 'mean'),
    return_rate_pct = ('status', lambda x: round((x=='Returned').sum()/len(x)*100, 1))
).round(2).sort_values('total_revenue', ascending=False).reset_index()

order_intelligence = df_labelled[[
    'order_id','order_date','region','category','revenue',
    'rev_tier','priority_score','manager_note','rolling_7','rev_rank'
]].copy()

top_priorities = (df_labelled
    .sort_values(['priority_score','revenue'], ascending=False)
    .head(20)
    [['order_id','order_date','region','category','revenue',
      'rev_tier','priority_score','manager_note','rolling_7','rev_rank']]
)

with pd.ExcelWriter('shopease_q1_report.xlsx', engine='openpyxl') as writer:
    region_summary.to_excel(writer,       sheet_name='Executive Summary', index=False)
    revenue_pivot.to_excel(writer,         sheet_name='Revenue Pivot',    index=True)
    order_intelligence.to_excel(writer,    sheet_name='Order Intelligence', index=False)
    top_priorities.to_excel(writer,        sheet_name='Top Priorities',   index=False)

for sheet in ['Executive Summary','Revenue Pivot','Order Intelligence','Top Priorities']:
    s = pd.read_excel('shopease_q1_report.xlsx', sheet_name=sheet).shape
    print(f"{sheet}: {s}")



Executive Summary: (4, 5)
Revenue Pivot: (5, 7)
Order Intelligence: (200, 10)
Top Priorities: (20, 10)


#### D2 — Format the Executive Summary Sheet (8 pts)
Re-open `'shopease_q1_report.xlsx'` with `load_workbook()` and apply to
the `'Executive Summary'` sheet:

1. **Header row (row 1):** dark blue `#1F3864`, white bold Arial 11, center aligned
2. **`total_revenue` and `avg_revenue` columns:** number format `'#,##0.00'`
3. **`return_rate_pct` column:** number format `'0.0%'`
   - ⚠️ The values are already percentages (e.g. 22.4) — format as `'0.00'` not `'0.0%'`
     since `0.0%` multiplies by 100 in Excel
4. **Freeze pane at `'B2'`** (so both the header AND the region column stay visible)
5. **Auto-fit all column widths** (max content + 4 padding)

Save and read back. Print: `"Executive Summary formatted. Rows: X, Cols: Y"`


In [9]:
# D2 — Format Executive Summary Sheet (8 pts)
# from openpyxl import load_workbook
# wb = load_workbook('shopease_q1_report.xlsx')
# ws = wb['Executive Summary']
# Apply: header fill, font, alignment
# Apply: number formats on revenue cols
# Note: return_rate_pct is already a % number (e.g. 22.4) → use '0.00' not '0.0%'
# Apply: freeze_panes = 'B2'
# Apply: auto-fit column widths
# wb.save(...)
# Read back and print confirmation
from openpyxl import load_workbook
from openpyxl.styles import PatternFill, Font, Alignment
from openpyxl.utils import get_column_letter

wb = load_workbook('shopease_q1_report.xlsx')
ws = wb['Executive Summary']

hdr_fill = PatternFill('solid', fgColor='1F3864')
hdr_font = Font(bold=True, color='FFFFFF', name='Arial', size=11)

for cell in ws[1]:
    cell.fill = hdr_fill
    cell.font = hdr_font
    cell.alignment = Alignment(horizontal='center')

cols = region_summary.columns.tolist()
for row in ws.iter_rows(min_row=2):
    for cell in row:
        cl = get_column_letter(cell.column)
        ci = cell.column - 1  # 0-indexed
        if ci < len(cols):
            cn = cols[ci]
            if cn in ('total_revenue','avg_revenue'): cell.number_format = '#,##0.00'
            if cn == 'return_rate_pct':               cell.number_format = '0.00'

ws.freeze_panes = 'B2'

for col in ws.columns:
    max_len = max(len(str(c.value or '')) for c in col) + 4
    ws.column_dimensions[get_column_letter(col[0].column)].width = max_len

wb.save('shopease_q1_report.xlsx')

verify = pd.read_excel('shopease_q1_report.xlsx', sheet_name='Executive Summary')
print(f"Executive Summary formatted. Rows: {verify.shape[0]}, Cols: {verify.shape[1]}")
print(verify.to_string(index=False))



Executive Summary formatted. Rows: 4, Cols: 5
region  total_revenue  order_count  avg_revenue  return_rate_pct
  East      136967.46           47      2914.20             17.0
 North      133672.49           57      2345.13             22.8
 South      124059.92           52      2385.77             21.2
  West       98121.23           44      2230.03             15.9


---
## 📊 Section 4 — Scoring Rubric

| Task | Sub-Task | Pts | What Is Checked |
|------|----------|-----|-----------------|
| **A — Pivot** | A1 Revenue Pivot | 10 | Correct aggfunc, margins=True labelled 'TOTAL', rounded, NRA cites specific cell value |
| | A2 Return Rate Crosstab | 10 | normalize='index' × 100, rounded to 1dp, NRA cites highest-return region by name |
| **B — Window** | B1 Cumulative Revenue | 10 | cumsum grouped by region after sort by date, last row per region printed, NRA present |
| | B2 Rolling + Rank | 10 | rolling(7, min_periods=1) on region group, rank dense descending, top 5 printed correctly |
| **C — Apply** | C1 Priority Score | 10 | Named function (not one-liner), all 5 conditions correct, value_counts + top 3 printed |
| | C2 Manager Note | 10 | All 7 note variants correct, value_counts printed, NRA cites URGENT count |
| **D — Export** | D1 Four-Sheet Workbook | 12 | 4 sheets with correct names, region_summary includes return_rate_pct, read-back shapes correct |
| | D2 Format Executive Summary | 8 | Dark blue header, correct number formats, freeze at B2, auto-width, read-back printed |
| **★ Bonus** | See below | 10 | — |
| **TOTAL** | | **80 + 10★** | |

---

### ⭐ Bonus Task — Conditional Row Highlight in 'Top Priorities' Sheet (10★ pts)

Re-open `'shopease_q1_report.xlsx'` with `load_workbook()` and open the `'Top Priorities'` sheet.

Apply row-level background colour to **every data row** based on `priority_score`:

| priority_score | Fill colour | Hex |
|---------------|-------------|-----|
| ≥ 7 | Red | `FFC7CE` |
| 5 or 6 | Orange | `FFEB9C` |
| 3 or 4 | Yellow | `FFFDB7` |
| ≤ 2 | No fill | — |

- Find the `priority_score` column index from the header row dynamically (no hardcoding column numbers)
- Loop through all data rows (row 2 onwards)
- Apply fill to the **entire row** (all columns), not just the score cell
- Save and print how many rows received each colour

> **Constraint:** You may not use `.conditional_formatting` — apply fills manually via openpyxl `iter_rows`.


In [18]:
# ⭐ BONUS — Row Highlight by Priority Score (10★ pts)
# 1. load_workbook('shopease_q1_report.xlsx')
# 2. ws = wb['Top Priorities']
# 3. Find priority_score column index from ws[1] dynamically
# 4. Loop ws.iter_rows(min_row=2), read priority_score cell value
# 5. Apply PatternFill to every cell in that row based on score
# 6. wb.save(...)
# 7. Print count of each colour applied
from openpyxl import load_workbook
from openpyxl.styles import PatternFill

wb = load_workbook('shopease_q1_report.xlsx')
ws = wb['Top Priorities']

# Dynamically find priority_score column index from header row
header = [cell.value for cell in ws[1]]
score_col_idx = header.index('priority_score') + 1    # 1‑based

# Define fills
red_fill    = PatternFill('solid', fgColor='FFC7CE')
orange_fill = PatternFill('solid', fgColor='FFEB9C')
yellow_fill = PatternFill('solid', fgColor='FFFDB7')

color_counts = {'red': 0, 'orange': 0, 'yellow': 0, 'none': 0}

for row in ws.iter_rows(min_row=2, max_row=ws.max_row):
    score_cell = row[score_col_idx - 1]
    score = score_cell.value

    if score >= 7:
        fill = red_fill
        color_counts['red'] += 1
    elif score in (5, 6):
        fill = orange_fill
        color_counts['orange'] += 1
    elif score in (3, 4):
        fill = yellow_fill
        color_counts['yellow'] += 1
    else:
        fill = None
        color_counts['none'] += 1

    if fill:
        for cell in row:
            cell.fill = fill

wb.save('shopease_q1_report.xlsx')

for color, count in color_counts.items():
    print(f"{color}: {count} rows")


red: 2 rows
orange: 11 rows
yellow: 7 rows
none: 0 rows


---

### Key Takeaway
A mini-project is not four separate exercises stapled together — it is one business question answered in four layers. The pivot tells you *where* the problem is. The window function tells you *when* it emerged. The apply/lambda tells you *what to do* about each order. The export delivers it in a form the decision-maker can act on without asking follow-up questions. This is the structure of every real analytics deliverable you will send to a client.

### Interview Frame
> "I structure analysis reports in four layers: a pivot summary for the big-picture breakdown, window functions for trend and ranking, row-level logic flags for triage priority, and a formatted multi-sheet Excel export as the final deliverable. The analysis tells the story — the export makes sure it gets read."

---
